# InsectAI WG3 Datathon - Template for data processing and validation

In [1]:
import os
import numpy as np
import pandas as pd
import json
from datetime import datetime, timezone
from PIL import Image
from PIL.ExifTags import TAGS
import mimetypes

## Loading data

In [2]:
with open("raw_labels/detections_val_20.json", "r") as f:
    data = json.load(f)

In [3]:
print(data.keys())

dict_keys(['info', 'licenses', 'categories', 'images', 'annotations'])


In [4]:
print(data['images'][0])

print(data['annotations'][0])

{'id': 3, 'file_name': 'C01_01_0_1606_1.jpg', 'width': 1024, 'height': 1024}
{'id': 6, 'image_id': 8, 'category_id': 2, 'category_name': 'bumblebee+', 'segmentation': [], 'area': 31040, 'bbox': [376, 336, 160, 194], 'iscrowd': 0, 'detectionID': 'C01_01_0_2902_517540_14', 'flash': 0, 'flower_species': 'scacol', 'focus': 'clear', 'scalingFactor': 213.0, 'deploymentID': '2023_CH_C01_01', 'sceneID': 'C01_01_0', 'mediaID': 'C01_01_0_2902', 'eventID': '517540_14', 'DOY': 173, 'Hour': 12, 'Minute': 35, 'lifeStage': 'adult', 'behaviour': 'on_flower'}


In [5]:
images = pd.DataFrame(data.get("images", [])).set_index("id")

anns = pd.DataFrame(data.get("annotations", [])).set_index("id")

anns.drop(columns=["segmentation", "area", "iscrowd"], inplace=True)

In [6]:
images.sample(5)

,file_name,width,height
id,,,
1469,HE07_01_32_5261_2.jpg,1024,1024
2293,HE21_02_33_8170_2.jpg,1024,1024
52,C01_01_2_199_2.jpg,1024,1024
113,C02_01_5_671_2.jpg,1024,1024
2243,HE21_02_16_10811_1.jpg,1024,1024


In [7]:
anns.sample(5)

,image_id,category_id,category_name,bbox,detectionID,flash,flower_species,focus,scalingFactor,deploymentID,sceneID,mediaID,eventID,DOY,Hour,Minute,lifeStage,behaviour
id,,,,,,,,,,,,,,,,,,
1033,1741,2,bumblebee+,"[477, 361, 280, 216]",HE10_02_11_7901_887617_9865,0,sucpra,clear,374.5,2023_NO_HE10_02,HE10_02_11,HE10_02_11_7901,887617_9865,216,11,45,adult,hovering
1050,1757,6,syrphid-,"[575, 579, 163, 151]",HE11_02_12_5094_668678_5380,0,sucpra,acceptable,280.0,2023_NO_HE11_02,HE11_02_12,HE11_02_12_5094,668678_5380,255,12,57,adult,on_flower
1325,2162,6,syrphid-,"[85, 280, 103, 64]",HE20_01_69_3103_803843_513,0,tripra,acceptable,154.5,2023_NO_HE20_01,HE20_01_69,HE20_01_69_3103,803843_513,172,16,33,adult,on_flower
1225,2015,7,diptera,"[275, 483, 127, 83]",HE15_02_14_7894_904816_10551,0,sucpra,clear,517.5,2023_NO_HE15_02,HE15_02_14,HE15_02_14_7894,904816_10551,210,20,47,adult,on_flower
1537,2465,8,lepidoptera,"[41, 621, 534, 323]",HE24_02_18_8942_754005_6902,1,sucpra,acceptable,317.0,2023_NO_HE24_02,HE24_02_18,HE24_02_18_8942,754005_6902,255,5,7,adult,on_flower


## Standardizing `deployments`

Example here: https://github.com/tdwg/camtrap-dp/blob/1.0.2/example/deployments.csv

### Changes needed

- **Renaming** `baitUse` to `attractantUse`
- **Adding** `attractantType` column

### Mandatory fields in `deployments`:
- `deploymentID`
- `latitude`
- `longitude`
- `deploymentStart`
- `deploymentEnd`

### Notes

For this specific example, I've used the `sceneID` as `deploymentID` for simplicity, but in a real-world scenario, you might want to create **unique deployment IDs** that are not tied to the scene ID, especially if multiple deployments can occur at the same location over time.

In [8]:
df_deployments = pd.read_csv("../../deployments_template.csv")

df_deployments.head()

,deploymentID,locationID,locationName,latitude,longitude,coordinateUncertainty,deploymentStart,deploymentEnd,setupBy,cameraID,...,cameraTilt,cameraHeading,detectionDistance,timestampIssues,baitUse,featureType,habitat,deploymentGroups,deploymentTags,deploymentComments


In [9]:
scene_ids = (images['file_name']
             .dropna()
             .astype(str)
             .map(lambda s: '_'.join(s.split('_')[:3]))
             .unique())
rows_to_add = pd.DataFrame({'deploymentID': scene_ids})

# safety check: if rows_to_add is injecting existing deploymentIDs in df_deployments
existing_deploymentIDs = set(df_deployments['deploymentID'])
rows_to_add = rows_to_add[~rows_to_add['deploymentID'].isin(existing_deploymentIDs)]

# reorder columns to match df_deployments filling with NaN values
rows_to_add = rows_to_add.reindex(columns=df_deployments.columns) 

df_deployments = pd.concat([df_deployments, rows_to_add], ignore_index=True)

print(f"Number of rows in df_deployments: {len(df_deployments)}")
df_deployments.head()

Number of rows in df_deployments: 111


,deploymentID,locationID,locationName,latitude,longitude,coordinateUncertainty,deploymentStart,deploymentEnd,setupBy,cameraID,...,cameraTilt,cameraHeading,detectionDistance,timestampIssues,baitUse,featureType,habitat,deploymentGroups,deploymentTags,deploymentComments
0,C01_01_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C01_01_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C01_02_4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C02_01_5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,C04_01_6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# NOTE: for simplicity, fixed deployment start and end dates, and fixed latitude and longitude values
start = datetime(2023, 1, 1, 0, 0, 0, tzinfo=timezone.utc)
end = datetime(2023, 12, 31, 23, 59, 59, tzinfo=timezone.utc)

df_deployments['deploymentStart'] = start.isoformat()
df_deployments['deploymentEnd'] = end.isoformat()
df_deployments['latitude'] = 0.0
df_deployments['longitude'] = 0.0

In [11]:
df_deployments.sample(5)

,deploymentID,locationID,locationName,latitude,longitude,coordinateUncertainty,deploymentStart,deploymentEnd,setupBy,cameraID,...,cameraTilt,cameraHeading,detectionDistance,timestampIssues,baitUse,featureType,habitat,deploymentGroups,deploymentTags,deploymentComments
61,HE07_02_5,NaN,NaN,0.0,0.0,NaN,2023-01-01T00:00:00+00:00,2023-12-31T23:59:59+00:00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,C14_01_28,NaN,NaN,0.0,0.0,NaN,2023-01-01T00:00:00+00:00,2023-12-31T23:59:59+00:00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,HE12_01_52,NaN,NaN,0.0,0.0,NaN,2023-01-01T00:00:00+00:00,2023-12-31T23:59:59+00:00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
88,HE17_01_66,NaN,NaN,0.0,0.0,NaN,2023-01-01T00:00:00+00:00,2023-12-31T23:59:59+00:00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,C14_01_30,NaN,NaN,0.0,0.0,NaN,2023-01-01T00:00:00+00:00,2023-12-31T23:59:59+00:00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Checking `deployments` for missing mandatory fields

In [12]:
# safety check: verify that all mandatory fields are present in df_deployments and count missing values
missing_deploymentID = df_deployments['deploymentID'].isna().sum()
missing_deploymentStart = df_deployments['deploymentStart'].isna().sum()
missing_deploymentEnd = df_deployments['deploymentEnd'].isna().sum()
missing_latitude = df_deployments['latitude'].isna().sum()
missing_longitude = df_deployments['longitude'].isna().sum()

print(f"Missing deploymentID: {missing_deploymentID}")
print(f"Missing deploymentStart: {missing_deploymentStart}")
print(f"Missing deploymentEnd: {missing_deploymentEnd}")
print(f"Missing latitude: {missing_latitude}")
print(f"Missing longitude: {missing_longitude}")

Missing deploymentID: 0
Missing deploymentStart: 0
Missing deploymentEnd: 0
Missing latitude: 0
Missing longitude: 0


## Standardizing `media`

Example here: https://github.com/tdwg/camtrap-dp/blob/1.0.2/example/media.csv

No changes needed

### Mandatory fields in `media`:
- `mediaID`
- `deploymentID`
- `timestamp`
- `filePath`
- `filePublic`
- `fileMediaType`

In [13]:
df_media = pd.read_csv("../../media_template.csv")

df_media.head()

,mediaID,deploymentID,captureMethod,timestamp,filePath,filePublic,fileName,fileMediatype,exifData,favorite,mediaComments


### Extracting `timestamp` and `fileMediatype`

**NOTE:** In this case, I had to extract the `timestamp` from the EXIF metadata of the images, as it was not provided in the original dataset. If the timestamp is already provided or formatted in the filename, you can skip this step and directly use the provided timestamp. Same thing for the `fileMediaType`.

Also the `deploymentID` can be extracted in different ways depending on how is structured the original dataset (e.g., filename, sub-folders name, etc.). In this case, I've extracted it from the `filename`.

In general, it is always useful to **extract the EXIF metadata** from the images, as it can contain useful information and camtrapDP provide a specific field for it (`exifData`).

In [14]:
data_dir = "media"

for idx, row in images.iterrows():
    media_id = idx
    file_path = row['file_name']
    deployment_id = '_'.join(file_path.split('_')[:3])
    file_public = True

    metadata = {}
    timestamp = None
    file_media_type = None

    try:
        file_path_full = os.path.join(data_dir, file_path)
        img = Image.open(file_path_full)
        exif_data = img._getexif()

        if exif_data is not None:
            for tag_id, value in exif_data.items():
                tag_name = TAGS.get(tag_id, tag_id)
                
                if isinstance(value, (bytes, bytearray)):
                    continue
                
                if isinstance(value, float) and np.isnan(value):
                    value = None

                if tag_name == 'DateTimeOriginal':
                    timestamp = value
                
                metadata[tag_name] = value
        else:
            print(f"No EXIF data for {file_path}")

        file_media_type, _ = mimetypes.guess_type(file_path_full)
        if not file_media_type:
            # fallback in case mimetypes fails
            file_media_type = "image/jpeg"

    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
    
    if timestamp:
        try:
            timestamp = datetime.strptime(timestamp, "%Y:%m:%d %H:%M:%S").astimezone(timezone.utc).isoformat()
        except ValueError:
            print(f"Invalid timestamp format for {file_path}: {timestamp}")
            timestamp = None

    exif_json = {"EXIF": metadata} if metadata else np.nan
    exif_str = json.dumps(exif_json, default=str)

    media_row = pd.DataFrame({
        'mediaID': [media_id],
        'deploymentID': [deployment_id],
        'timestamp': [timestamp],
        'filePath': [file_path_full],
        'filePublic': [file_public],
        'fileMediatype': [file_media_type],
        'exifData': [exif_str]
    })
    
    # safety check to avoid duplicates
    if media_id not in df_media['mediaID'].values:
        media_row = media_row.reindex(columns=df_media.columns)
        df_media = pd.concat([df_media, media_row], ignore_index=True)
    else:
        print(f"Skipping duplicate mediaID {media_id}")

In [15]:
df_media.sample(5)

,mediaID,deploymentID,captureMethod,timestamp,filePath,filePublic,fileName,fileMediatype,exifData,favorite,mediaComments
120,626,C17_01_37,NaN,2023-07-16T08:09:51+00:00,media/C17_01_37_9246_2.jpg,True,NaN,image/jpeg,"{""EXIF"": {""GPSInfo"": {""0"": ""b'\\x02\\x02\\x00\...",NaN,NaN
296,1559,HE08_01_39,NaN,2023-07-11T07:46:10+00:00,media/HE08_01_39_9222_1.jpg,True,NaN,image/jpeg,"{""EXIF"": {""GPSInfo"": {""0"": ""b'\\x02\\x02\\x00\...",NaN,NaN
505,2529,HE24_02_40,NaN,2023-09-17T13:14:11+00:00,media/HE24_02_40_8112_1.jpg,True,NaN,image/jpeg,"{""EXIF"": {""GPSInfo"": {""0"": ""b'\\x02\\x02\\x00\...",NaN,NaN
351,1771,HE11_02_22,NaN,2023-08-31T01:16:07+00:00,media/HE11_02_22_196_2.jpg,True,NaN,image/jpeg,"{""EXIF"": {""GPSInfo"": {""0"": ""b'\\x02\\x02\\x00\...",NaN,NaN
364,1818,HE12_01_47,NaN,2023-07-09T18:00:43+00:00,media/HE12_01_47_2641_1.jpg,True,NaN,image/jpeg,"{""EXIF"": {""GPSInfo"": {""0"": ""b'\\x02\\x02\\x00\...",NaN,NaN


### Checking `media` for missing mandatory fields

In [16]:
# safety check: verify that all mandatory fields are present in df_media and count missing values
missing_mediaID = df_media['mediaID'].isnull().sum()
missing_deploymentID = df_media['deploymentID'].isnull().sum()
missing_timestamp = df_media['timestamp'].isnull().sum()
missing_filePath = df_media['filePath'].isnull().sum()
missing_filePublic = df_media['filePublic'].isnull().sum()
missing_fileMediaType = df_media['fileMediatype'].isnull().sum()

print(f"Missing mediaID: {missing_mediaID}")
print(f"Missing deploymentID: {missing_deploymentID}")
print(f"Missing timestamp: {missing_timestamp}")
print(f"Missing filePath: {missing_filePath}")
print(f"Missing filePublic: {missing_filePublic}")
print(f"Missing fileMediaType: {missing_fileMediaType}")

# safety check: verify that all deploymentIDs in df_media are present in df_deployments
media_deploymentIDs = set(df_media['deploymentID'].dropna().unique())
deployments_deploymentIDs = set(df_deployments['deploymentID'].dropna().unique())
missing_in_deployments = media_deploymentIDs - deployments_deploymentIDs
if missing_in_deployments:
    print(f"WARNING: The following deploymentIDs are in df_media but missing in df_deployments: {missing_in_deployments}")

Missing mediaID: 0
Missing deploymentID: 0
Missing timestamp: 0
Missing filePath: 0
Missing filePublic: 0
Missing fileMediaType: 0


### Additional checks for `media`

In addition to checking for missing mandatory fields, it can be useful to perform some additional checks on the `media` dataset, depending on the original dataset and from some issues that you have might encountered during the field work or during the data processing. For example, in this case we have some wrong metadata for some images caused by a battery out-age during the field work, which caused the camera to reset and lose the correct timestamp.

The camtrapDP format provide a `timestampIssues` field in the `deployments` dataset that can be used to flag any issues related to the timestamp of the media. So, we may want to set it as `True` for deployments containing media with wrong timestamps.

In [17]:
# for each deploymentID in df_deployments, check if there are media with timestamp issues, so timestamp out of deployment start and end dates
wrong_timestamps = []

for deployment_id in df_deployments['deploymentID'].dropna().unique():
    deployment_row = df_deployments[df_deployments['deploymentID'] == deployment_id]
    deployment_start = deployment_row['deploymentStart'].values[0]
    deployment_end = deployment_row['deploymentEnd'].values[0]

    media_rows = df_media[df_media['deploymentID'] == deployment_id]
    for idx, media_row in media_rows.iterrows():
        timestamp = media_row['timestamp']
        if pd.isnull(timestamp):
            continue
        if timestamp < deployment_start or timestamp > deployment_end:
            print(f"Timestamp issue for mediaID {media_row['mediaID']} in deploymentID {deployment_id}: timestamp {timestamp} is out of deployment start and end dates")
            df_deployments.loc[df_deployments['deploymentID'] == deployment_id, 'timestampIssues'] = True
            wrong_timestamps.append(media_row['mediaID'])
        else:
            df_deployments.loc[df_deployments['deploymentID'] == deployment_id, 'timestampIssues'] = False

print(f"Total media with timestamp issues: {len(wrong_timestamps)}")

Timestamp issue for mediaID 77 in deploymentID C01_02_4: timestamp 2016-11-17T01:46:22+00:00 is out of deployment start and end dates
Timestamp issue for mediaID 78 in deploymentID C01_02_4: timestamp 2016-11-17T01:47:22+00:00 is out of deployment start and end dates
Timestamp issue for mediaID 87 in deploymentID C01_02_4: timestamp 2016-11-20T16:47:39+00:00 is out of deployment start and end dates
Timestamp issue for mediaID 88 in deploymentID C01_02_4: timestamp 2016-11-20T16:48:39+00:00 is out of deployment start and end dates
Timestamp issue for mediaID 95 in deploymentID C01_02_4: timestamp 2016-11-21T23:22:41+00:00 is out of deployment start and end dates
Timestamp issue for mediaID 576 in deploymentID C16_02_34: timestamp 2016-11-14T23:47:42+00:00 is out of deployment start and end dates
Timestamp issue for mediaID 590 in deploymentID C16_02_34: timestamp 2016-11-18T22:59:58+00:00 is out of deployment start and end dates
Timestamp issue for mediaID 592 in deploymentID C16_02_34:

In [18]:
df_deployments.head()

,deploymentID,locationID,locationName,latitude,longitude,coordinateUncertainty,deploymentStart,deploymentEnd,setupBy,cameraID,...,cameraTilt,cameraHeading,detectionDistance,timestampIssues,baitUse,featureType,habitat,deploymentGroups,deploymentTags,deploymentComments
0,C01_01_0,NaN,NaN,0.0,0.0,NaN,2023-01-01T00:00:00+00:00,2023-12-31T23:59:59+00:00,NaN,NaN,...,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN
1,C01_01_2,NaN,NaN,0.0,0.0,NaN,2023-01-01T00:00:00+00:00,2023-12-31T23:59:59+00:00,NaN,NaN,...,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN
2,C01_02_4,NaN,NaN,0.0,0.0,NaN,2023-01-01T00:00:00+00:00,2023-12-31T23:59:59+00:00,NaN,NaN,...,NaN,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN
3,C02_01_5,NaN,NaN,0.0,0.0,NaN,2023-01-01T00:00:00+00:00,2023-12-31T23:59:59+00:00,NaN,NaN,...,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN
4,C04_01_6,NaN,NaN,0.0,0.0,NaN,2023-01-01T00:00:00+00:00,2023-12-31T23:59:59+00:00,NaN,NaN,...,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN


## Standardizing `observations`

Example here: https://github.com/tdwg/camtrap-dp/blob/1.0.2/example/observations.csv

No changes needed

### Mandatory fields in `observations`:
- `observationID`
- `deploymentID`
- `eventStart`
- `eventEnd`
- `observationLevel`
- `observationType`

In [19]:
df_observations = pd.read_csv("../../observations_template.csv")

df_observations.head()

,observationID,deploymentID,mediaID,eventID,eventStart,eventEnd,observationLevel,observationType,cameraSetupType,scientificName,...,bboxX,bboxY,bboxWidth,bboxHeight,classificationMethod,classifiedBy,classificationTimestamp,classificationProbability,observationTags,observationComments


### Notes

#### Extracting `scientificName`
This step is not strictly necessary, it depends on how the categories were mapped in the original dataset. In this case, I had to map the original categories to the corresponding scientific names, as the original dataset only provided common names or category ids. If the original dataset already provides the scientific names, you can skip this step and directly use the provided scientific names.

#### Additional metadata
The original dataset may contain additional metadata that can be useful to include in the `observations` dataset, but without a corresponding field in the camtrapDP format. In this case, I had some additional metadata such as the *flash status*, *flower species*, *camera focus* information and *scalingFactor*. As stated in the camtrapDP documentation, you can include any additional metadata in the `observationTags` field.

In [20]:
cat_to_sci = {
    2: "Bombus", # bumblebee
    3: "Apis mellifera", # bee
    4: "Formicidae", # ant
    5: "Hymenoptera", # hymenoptera
    6: "Syrphidae", # syrphid
    7: "Diptera", # diptera
    8: "Lepidoptera", # lepidoptera
    9: "Coleoptera" # coleoptera
}

In [21]:
for idx, row in anns.iterrows():
    observation_id = idx
    deployment_id = row['sceneID']
    media_id = row['image_id']
    event_id = row['eventID']

    DOY = row['DOY']
    hour = row['Hour']
    minute = row['Minute']
    event_time = datetime(2023, 1, 1) + pd.to_timedelta(DOY - 1, unit='D') + pd.to_timedelta(hour, unit='h') + pd.to_timedelta(minute, unit='m')
    event_time = event_time.replace(tzinfo=timezone.utc).isoformat()

    observation_level = "media"
    observation_type = "animal"
    scientific_name = cat_to_sci.get(row['category_id'], pd.NA)
    count = 1  # each annotation corresponds to one observation
    lifestage = row.get('lifeStage', np.nan)
    behavior = row.get('behavior', np.nan)

    bboxX, bboxY, bboxWidth, bboxHeight = row['bbox']
    # camtrapDP expects normalized bbox coordinates
    img = Image.open(os.path.join(data_dir, images.loc[media_id, 'file_name']))
    img_width, img_height = img.size
    bboxX = np.clip(bboxX / img_width, 0, 1)
    bboxY = np.clip(bboxY / img_height, 0, 1)
    bboxWidth = np.clip(bboxWidth / img_width, 0, 1)
    bboxHeight = np.clip(bboxHeight / img_height, 0, 1)

    classification_method = "human"

    observation_tags = []
    for key in ('flash', 'flower_species', 'focus', 'scalingFactor'):
        val = row.get(key, None)
        if pd.notna(val):
            observation_tags.append(f"{key}:{val}")
    observation_tags = "|".join(observation_tags) if observation_tags else pd.NA

    observation_row = pd.DataFrame({
        'observationID': [observation_id],
        'deploymentID': [deployment_id],
        'mediaID': [media_id],
        'eventID': [event_id],
        'eventStart': [event_time],
        'eventEnd': [event_time],
        'observationLevel': [observation_level],
        'observationType': [observation_type],
        'scientificName': [scientific_name],
        'count': [count],
        'lifestage': [lifestage],
        'behavior': [behavior],
        'bboxX': [bboxX],
        'bboxY': [bboxY],
        'bboxWidth': [bboxWidth],
        'bboxHeight': [bboxHeight],
        'classificationMethod': [classification_method],
        'observationTags': [observation_tags]
    })
    
    # safety check to avoid duplicates
    if observation_id not in df_observations['observationID'].values:
        observation_row = observation_row.reindex(columns=df_observations.columns)
        df_observations = pd.concat([df_observations, observation_row], ignore_index=True)
    else:
        print(f"Skipping duplicate observationID {observation_id}")

In [22]:
df_observations.sample(5)

,observationID,deploymentID,mediaID,eventID,eventStart,eventEnd,observationLevel,observationType,cameraSetupType,scientificName,...,bboxX,bboxY,bboxWidth,bboxHeight,classificationMethod,classifiedBy,classificationTimestamp,classificationProbability,observationTags,observationComments
269,1314,HE19_01_68,2155,802140_7808,2023-07-05T15:58:00+00:00,2023-07-05T15:58:00+00:00,media,animal,NaN,Lepidoptera,...,0.366211,0.0,0.233398,0.289062,human,NaN,NaN,NaN,flash:0|flower_species:tripra|focus:blur|scali...,NaN
294,1422,HE22_01_83,2300,835061_8851,2023-06-23T09:58:00+00:00,2023-06-23T09:58:00+00:00,media,animal,NaN,Bombus,...,0.286133,0.303711,0.267578,0.230469,human,NaN,NaN,NaN,flash:0|flower_species:tripra|focus:acceptable...,NaN
90,489,C23_02_51,872,1253082_59,2023-08-02T12:36:00+00:00,2023-08-02T12:36:00+00:00,media,animal,NaN,Syrphidae,...,0.478516,0.52832,0.253906,0.262695,human,NaN,NaN,NaN,flash:0|flower_species:scacol|focus:blur|scali...,NaN
53,240,C13_01_26,438,668606_1192,2023-06-22T12:37:00+00:00,2023-06-22T12:37:00+00:00,media,animal,NaN,Formicidae,...,0.429688,0.397461,0.060547,0.060547,human,NaN,NaN,NaN,flash:0|flower_species:scacol|focus:clear|scal...,NaN
231,1123,HE12_02_13,1881,668681_5420,2023-09-06T10:35:00+00:00,2023-09-06T10:35:00+00:00,media,animal,NaN,Bombus,...,0.0,0.166992,0.360352,0.369141,human,NaN,NaN,NaN,flash:0|flower_species:sucpra|focus:blur|scali...,NaN


### Checking `observations` for missing mandatory fields

In [23]:
# safety check: verify that all mandatory fields are present in df_observations and count missing valuesa
missing_mediaID = df_observations['observationID'].isnull().sum()
missing_deploymentID = df_observations['deploymentID'].isnull().sum()
missing_eventStart = df_observations['eventStart'].isnull().sum()
missing_eventEnd = df_observations['eventEnd'].isnull().sum()
missing_observationLevel = df_observations['observationLevel'].isnull().sum()
missing_observationType = df_observations['observationType'].isnull().sum()

print(f"Missing observationID: {missing_mediaID}")
print(f"Missing deploymentID: {missing_deploymentID}")
print(f"Missing eventStart: {missing_eventStart}")
print(f"Missing eventEnd: {missing_eventEnd}")
print(f"Missing observationLevel: {missing_observationLevel}")
print(f"Missing observationType: {missing_observationType}")

# safety check: verify that all deploymentIDs in df_media are present in df_deployments
media_deploymentIDs = set(df_media['deploymentID'].dropna().unique())
deployments_deploymentIDs = set(df_deployments['deploymentID'].dropna().unique())
missing_in_deployments = media_deploymentIDs - deployments_deploymentIDs
if missing_in_deployments:
    print(f"WARNING: The following deploymentIDs are in df_media but missing in df_deployments: {missing_in_deployments}")

# safety check: verify that all mediaIDs in df_observations are present in df_media
observations_mediaIDs = set(df_observations['mediaID'].dropna().unique())
media_mediaIDs = set(df_media['mediaID'].dropna().unique())
missing_in_media = observations_mediaIDs - media_mediaIDs
if missing_in_media:
    print(f"WARNING: The following mediaIDs are in df_observations but missing in df_media: {missing_in_media}")

Missing observationID: 0
Missing deploymentID: 0
Missing eventStart: 0
Missing eventEnd: 0
Missing observationLevel: 0
Missing observationType: 0


## Standardizing `datapackage.json`

Example here: https://github.com/tdwg/camtrap-dp/blob/1.0.2/example/datapackage.json

No changes needed

### Mandatory fields in `datapackage.json`:
- `resources` (with FOR EACH ONE `name`, `path`, `profile` and `schema`)
- `profile`
- `created`
- `contributors`
- `project` (with `title`, `samplingDesign`, `captureMethod`, `individualAnimals`, `observationLevel`)
- `spatial`
- `temporal` (with `start` and `end`)
- `taxonomic` (with FOR EACH ONE `scientificName`)

### Notes

As regard the `taxonomic` field, in this case was easier adding the scientific names in the taxonomic section thanks to the mapping. However, if the original dataset already provides the scientific names in the `observations` dataset, you can directly use those scientific names in the `taxonomic` section of the `datapackage.json`, without needing to insert them manually. The important thing is that the `scientificName` field in the `taxonomic` section should contain all the unique scientific names that are present in the `observations` dataset, so that they can be properly linked and referenced.

In [24]:
# MANDATORY
datapackage_resources = [
{
    "name": "deployments",
    "path": "deployments.csv",
    "profile": "tabular-data-resource",
    "format": "csv",
    "mediatype": "text/csv",
    "encoding": "utf-8",
    "schema": "https://raw.githubusercontent.com/tdwg/camtrap-dp/1.0/deployments-table-schema.json"
},
{
    "name": "media",
    "path": "media.csv",
    "profile": "tabular-data-resource",
    "format": "csv",
    "mediatype": "text/csv",
    "encoding": "utf-8",
    "schema": "https://raw.githubusercontent.com/tdwg/camtrap-dp/1.0/media-table-schema.json" 
},
{
    "name": "observations",
    "path": "observations.csv",
    "profile": "tabular-data-resource",
    "format": "csv",
    "mediatype": "text/csv",
    "encoding": "utf-8",
    "schema": "https://raw.githubusercontent.com/tdwg/camtrap-dp/1.0/observations-table-schema.json"
}]

datapackage = {
    "resources": datapackage_resources,
    "name": "rangex-coco-insectai-datathon",
    "created": datetime.now(timezone.utc).isoformat(),
    "contributors": [{
        "title": "Jamie Alison",
        "role": "author"
    }],
    "licenses": [{
        "scope": "media", # enum ("data", "media")
        "name": "CC-BY-4.0",
        "path": "https://creativecommons.org/licenses/by/4.0/",
        "title": "Creative Commons Attribution 4.0 International"
        # NOTE: or a more restrictive license
        # {
        #     "name": "CC-BY-NC-4.0",
        #     "path": "https://creativecommons.org/licenses/by-nc/4.0/",
        #     "title": "Creative Commons Attribution-NonCommercial 4.0 International"
        # }
    }],
    "project": {
        "title": "RangeX COCO InsectAI Datathon",
        "samplingDesign": "experimental", # enum (simpleRandom, systematicRandom, clusteredRandom, experimental, targeted, opportunistic)
        "captureMethod": "timeLapse",
        "individualAnimals": False,
        "observationLevel": "media"
    },
    "spatial": {
        # NOTE: random coordinates for simplicity
        "type": "Polygon",
        "coordinates": [[
            [4.013, 50.699],
            [5.659, 50.699],
            [5.659, 51.496],
            [4.013, 51.496],
            [4.013, 50.699]
        ]]
    },
    "temporal": {
        "start": "2023-01-01",
        "end": "2023-12-31"
    },
    "taxonomic": [{"scientificName": cat_to_sci[cat_id]} for cat_id in sorted(cat_to_sci.keys())]
}

### Checking `datapackage.json` for missing mandatory fields

In [25]:
for key in ['resources', 'created', 'contributors', 'project', 'spatial', 'temporal', 'taxonomic']:
    if key not in datapackage:
        raise ValueError(f"WARNING: Missing mandatory field '{key}' in datapackage.json")

for resource in datapackage['resources']:
    for key in ['name', 'path', 'profile', 'schema']:
        if key not in resource:
            print(f"WARNING: Missing mandatory field '{key}' in resource '{resource.get('name', 'unknown')}' in datapackage.json")

for key in ['title', 'samplingDesign', 'captureMethod', 'individualAnimals', 'observationLevel']:
    if key not in datapackage['project']:
        print(f"WARNING: Missing mandatory field '{key}' in 'project' section of datapackage.json")

for key in ['start', 'end']:
    if key not in datapackage['temporal']:
        print(f"WARNING: Missing mandatory field '{key}' in 'temporal' section of datapackage.json")

for taxon in datapackage['taxonomic']:
    if 'scientificName' not in taxon:
        print(f"WARNING: Missing mandatory field 'scientificName' in one of the entries in 'taxonomic' section of datapackage.json")

## Export dataframes and create `datapackage.json` file

In [26]:
df_deployments.to_csv("deployments.csv", index=False)
df_media.to_csv("media.csv", index=False)
df_observations.to_csv("observations.csv", index=False)

with open("datapackage.json", "w") as f:
    json.dump(datapackage, f, indent=4)

## Check dataset format correctness with the `frictionless` package

To check the dataset format correctness, you can use the `frictionless` package in Python. This package provides tools for validating and working with data packages, including checking for missing mandatory fields, validating data types, and ensuring that the dataset adheres to the specified schema.

In [27]:
!frictionless validate datapackage.json

─────────────────────────────────── Dataset ────────────────────────────────────
                      dataset                       
┏━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ name         ┃ type  ┃ path             ┃ status ┃
┡━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ deployments  │ table │ deployments.csv  │ VALID  │
│ media        │ table │ media.csv        │ VALID  │
│ observations │ table │ observations.csv │ VALID  │
└──────────────┴───────┴──────────────────┴────────┘
